# Encapsulation

An object keeps its data coherent only if nobody changes it behind its back.

In [1]:
class Programme:
    def __init__(self):
        self.shows = []

    def add_show(self, title, venue, start):
        for show in self.shows:
            if show["venue"] == venue and show["start"] == start:
                raise ValueError(f"{venue} is busy at {start}.")
        self.shows.append({"title": title, "venue": venue, "start": start})


programme = Programme()
programme.add_show("Opening Gala", "Grand Casino", "20:30")
programme.shows.append({"title": "Oops", "venue": "Grand Casino", "start": "20:30"})
print(len(programme.shows))

2


The rule of `add_show` was bypassed: two shows in the same venue at the same time.

## A leading underscore: private

In [2]:
class Programme:
    def __init__(self):
        self._shows = []

    def add_show(self, title, venue, start):
        for show in self._shows:
            if show["venue"] == venue and show["start"] == start:
                raise ValueError(f"{venue} is busy at {start}.")
        self._shows.append({"title": title, "venue": venue, "start": start})

    def shows(self):
        return list(self._shows)


programme = Programme()
programme.add_show("Opening Gala", "Grand Casino", "20:30")
programme.shows().append({"title": "Oops", "venue": "Grand Casino", "start": "20:30"})
print(len(programme.shows()))

1


`shows()` gives a copy: changing the copy does not change the programme.

`programme._shows` can still be reached: `_` is a convention, "not part of
what the class offers, do not use from outside". Everyone respects it.

## Read-only: a property without setter

In [3]:
class Wallet:
    def __init__(self, owner):
        self.owner = owner
        self._balance = 0.0

    @property
    def balance(self):
        return self._balance

    def top_up(self, amount):
        if amount <= 0:
            raise ValueError(f"Invalid amount: {amount}.")
        self._balance += amount


wallet = Wallet("Lina")
wallet.top_up(20)
print(wallet.balance)

20.0


In [4]:
wallet.balance = 1000

AttributeError: property 'balance' of 'Wallet' object has no setter

## A property with a setter: checked at each assignment

In [5]:
class Venue:
    def __init__(self, name, capacity):
        self.name = name
        self.capacity = capacity

    @property
    def capacity(self):
        return self._capacity

    @capacity.setter
    def capacity(self, value):
        if value < 1:
            raise ValueError(f"A venue needs at least one seat, got {value}.")
        self._capacity = value


kiosque = Venue("Le Kiosque", 120)
kiosque.capacity = 150
print(kiosque.capacity)

150


In [6]:
kiosque.capacity = 0

ValueError: A venue needs at least one seat, got 0.

`self.capacity = capacity` in `__init__` also goes through the setter: the check is written once.

## Two underscores

In [7]:
class Secret:
    def __init__(self):
        self.__code = "X7Q2"


print(vars(Secret()))

{'_Secret__code': 'X7Q2'}


Python renames `__code` into `_Secret__code`. Rarely needed: one underscore is
the convention.

| Data | Written as |
|---|---|
| free to read and change | `self.name` |
| read from outside, changed only by the class | `self._balance` + `@property` |
| changed from outside, but checked | `@property` + `@x.setter` |
| a list the class protects | `self._shows` + a method returning a copy |